# People aggregation: record linkage baseline

Use identifier-confirmed cross-source pairs to measure blocking and name-rule recall. This validation subset contains known positives, not confirmed negatives, so it cannot measure precision yet.

In [ ]:
import luigi
import pandas as pd
import recordlinkage

from pkb.tasks.build.people.identifier_matches import PeopleIdentifierMatchesTask

## Identifier validation pairs

In [ ]:
task = PeopleIdentifierMatchesTask()
luigi.build([task], local_scheduler=True)

pair_columns = [
    "left_source",
    "left_source_id",
    "right_source",
    "right_source_id",
]
identifier_matches = pd.read_csv(
    task.output().path,
    keep_default_na=False,
    dtype={column: "string" for column in [*pair_columns, "identifier_value"]},
)
pairs = identifier_matches.drop_duplicates(pair_columns).copy()
pairs.shape

In [ ]:
identifier_matches.groupby(
    ["left_source", "right_source", "identifier_type"]
).size().rename("matches").reset_index()

## Records participating in known matches

This keeps the first experiment small. Once the rules are useful, candidate generation can run over every source record.

In [ ]:
person_columns = [
    "source",
    "source_id",
    "name",
    "name_normalised",
    "surname",
    "given_names",
    "initials",
    "birth_date",
    "death_date",
    "geography",
]

left = pairs[[f"left_{column}" for column in person_columns]].copy()
left.columns = person_columns
right = pairs[[f"right_{column}" for column in person_columns]].copy()
right.columns = person_columns

people = pd.concat([left, right], ignore_index=True).drop_duplicates(
    ["source", "source_id"]
)
people["source"] = people["source"].astype("string")
people["source_id"] = people["source_id"].astype("string")
people["person_key"] = people["source"].str.cat(people["source_id"], sep=":")
people["birth_year"] = people["birth_date"].str.extract(r"(\d{4})", expand=False)
people["death_year"] = people["death_date"].str.extract(r"(\d{4})", expand=False)

for column in ["surname", "name_normalised", "initials", "birth_year", "death_year"]:
    people[column] = people[column].replace("", pd.NA)

people = people.set_index("person_key", drop=False)
people.shape

## Block on normalized surname

In [ ]:
blocked_people = people[people["surname"].notna()]

indexer = recordlinkage.Index()
indexer.block("surname")
candidate_links = indexer.index(blocked_people)
candidate_links = pd.MultiIndex.from_tuples(
    [
        (left_key, right_key)
        for left_key, right_key in candidate_links
        if blocked_people.at[left_key, "source"]
        != blocked_people.at[right_key, "source"]
    ],
    names=candidate_links.names,
)
len(candidate_links)

In [ ]:
pairs["left_person_key"] = pairs["left_source"].str.cat(pairs["left_source_id"], sep=":")
pairs["right_person_key"] = pairs["right_source"].str.cat(pairs["right_source_id"], sep=":")
pairs["pair_key"] = [
    frozenset((left_key, right_key))
    for left_key, right_key in zip(pairs["left_person_key"], pairs["right_person_key"])
]

known_pairs = set(pairs["pair_key"])
candidate_pairs = {frozenset(link) for link in candidate_links}
blocking_recall = len(known_pairs & candidate_pairs) / len(known_pairs)
blocking_recall

## Compare candidate pairs

In [ ]:
compare = recordlinkage.Compare()
compare.exact("name_normalised", "name_normalised", label="same_normalised_name")
compare.exact("initials", "initials", label="same_initials")
compare.exact("birth_year", "birth_year", label="same_birth_year")
compare.exact("death_year", "death_year", label="same_death_year")

features = compare.compute(candidate_links, blocked_people)
features.head()

In [ ]:
features["predicted_match"] = (
    features["same_normalised_name"].eq(1)
    | (
        features["same_initials"].eq(1)
        & (
            features["same_birth_year"].eq(1)
            | features["same_death_year"].eq(1)
        )
    )
)

predicted_links = features.index[features["predicted_match"]]
predicted_pairs = {frozenset(link) for link in predicted_links}
rule_recall = len(known_pairs & predicted_pairs) / len(known_pairs)

pd.DataFrame(
    [
        {"stage": "surname block", "recall": blocking_recall},
        {"stage": "comparison rule", "recall": rule_recall},
    ]
)

## False positives and false negatives

Known positives share an identifier. Hard negatives have known, conflicting values for the same identifier type. Other candidate pairs remain unlabelled and are excluded from false-positive metrics.

In [ ]:
evidence_left = identifier_matches[
    ["left_source", "left_source_id", "identifier_type", "identifier_value"]
].rename(columns={"left_source": "source", "left_source_id": "source_id"})
evidence_right = identifier_matches[
    ["right_source", "right_source_id", "identifier_type", "identifier_value"]
].rename(columns={"right_source": "source", "right_source_id": "source_id"})
identifier_evidence = pd.concat([evidence_left, evidence_right]).drop_duplicates()
identifier_evidence["person_key"] = identifier_evidence["source"].str.cat(
    identifier_evidence["source_id"], sep=":"
)
identifier_lookup = {
    key: set(values)
    for key, values in identifier_evidence.groupby(
        ["person_key", "identifier_type"]
    )["identifier_value"]
}
identifier_types = identifier_evidence["identifier_type"].unique()


def identifiers_conflict(left_key, right_key):
    for identifier_type in identifier_types:
        left_values = identifier_lookup.get((left_key, identifier_type), set())
        right_values = identifier_lookup.get((right_key, identifier_type), set())
        if left_values and right_values and left_values.isdisjoint(right_values):
            return True
    return False


hard_negative_pairs = {
    frozenset((left_key, right_key))
    for left_key, right_key in candidate_links
    if identifiers_conflict(left_key, right_key)
}
hard_negative_pairs -= known_pairs
len(hard_negative_pairs)

In [ ]:
true_positive = len(predicted_pairs & known_pairs)
false_negative = len(known_pairs - predicted_pairs)
false_positive = len(predicted_pairs & hard_negative_pairs)
true_negative = len(hard_negative_pairs - predicted_pairs)
unlabelled_predictions = len(
    predicted_pairs - known_pairs - hard_negative_pairs
)

pd.DataFrame(
    [
        {
            "true_positive": true_positive,
            "false_positive": false_positive,
            "true_negative": true_negative,
            "false_negative": false_negative,
            "labelled_precision": true_positive / (true_positive + false_positive),
            "recall": true_positive / (true_positive + false_negative),
            "false_positive_rate": false_positive / (false_positive + true_negative),
            "false_negative_rate": false_negative / (true_positive + false_negative),
            "unlabelled_predictions": unlabelled_predictions,
        }
    ]
)

In [ ]:
pairs["candidate"] = pairs["pair_key"].isin(candidate_pairs)
pairs["predicted_match"] = pairs["pair_key"].isin(predicted_pairs)

pairs.groupby(["left_source", "right_source"]).agg(
    identifier_matches=("pair_key", "size"),
    candidates=("candidate", "sum"),
    predicted_matches=("predicted_match", "sum"),
    blocking_recall=("candidate", "mean"),
    rule_recall=("predicted_match", "mean"),
).reset_index()

## Inspect known matches that were missed

In [ ]:
pairs["miss_stage"] = "comparison rule"
pairs.loc[~pairs["candidate"], "miss_stage"] = "surname block"

pairs.loc[
    ~pairs["predicted_match"],
    [
        "miss_stage",
        "left_source",
        "left_source_id",
        "left_name",
        "right_source",
        "right_source_id",
        "right_name",
        "left_birth_date",
        "right_birth_date",
        "identifier_type",
        "identifier_value",
    ],
].head(50)